## Fire & Prescribed Burn Map

In [1]:
# Import Packages
import pandas as pd
import geopandas as gpd
import folium
import pathlib as path

In [2]:
# Set up paths
root_path = path.Path.cwd().parent
raw_fldr = root_path / "data" / "raw"

In [3]:
# Load shapefiles
fire_gdf = gpd.read_file(raw_fldr / "fire25.zip")
rxburn_gdf = gpd.read_file(raw_fldr / "rxburn25.zip")

# Replace cause codes with labels
cause_lookup = pd.read_csv(raw_fldr / "fire_25_TD.csv", usecols=["CAUSE", "d_CAUSE"]).drop_duplicates().set_index("CAUSE")["d_CAUSE"]
fire_gdf["CAUSE"] = fire_gdf["CAUSE"].map(cause_lookup)

# Fire duration in days
fire_gdf["DURATION_DAYS"] = (fire_gdf["CONT_DATE"] - fire_gdf["ALARM_DATE"]).dt.days

# Keep 2000+ and simplify shapes (100 m) so the map stays fast
fire_gdf = fire_gdf[fire_gdf["YEAR_"] >= 2000]
rxburn_gdf = rxburn_gdf[rxburn_gdf["YEAR_"] >= 2000]
fire_gdf["geometry"] = fire_gdf.geometry.simplify(100)
rxburn_gdf["geometry"] = rxburn_gdf.geometry.simplify(100)

/opt/homebrew/Caskroom/miniforge/base/envs/geo/lib/python3.11/site-packages/pyogrio/raw.py:200: RuntimeWarning: organizePolygons() received an unexpected geometry.  Either a polygon with interior rings, or a polygon with less than 4 points, or a non-Polygon geometry.  Return arguments as a collection.
  return ogr_read(
/opt/homebrew/Caskroom/miniforge/base/envs/geo/lib/python3.11/site-packages/pyogrio/raw.py:200: RuntimeWarning: Geometry of polygon of fid 3575 cannot be translated to Simple Geometry. All polygons will be contained in a multipolygon.
  return ogr_read(


In [4]:
def make_fire_map(fires, rx, start_date, end_date, causes=None, show_fires=True, show_rx=True):
    start, end = pd.to_datetime(start_date), pd.to_datetime(end_date)
    fires = fires[fires["ALARM_DATE"].between(start, end)]
    rx = rx[rx["START_DATE"].between(start, end)]
    if causes is not None:
        fires = fires[fires["CAUSE"].isin(causes)]

    m = folium.Map(
        location=[37.2, -119.5], zoom_start=6,
        tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Topo_Map/MapServer/tile/{z}/{y}/{x}",
        attr="Esri",
    )

    if show_fires:
        fires.explore(
            m=m, name="Fires", column="DURATION_DAYS", cmap="YlOrRd", vmax=60,
            missing_kwds={"color": "lightgrey"},
            legend_kwds={"caption": "Fire duration (days)"},
            tooltip=["FIRE_NAME", "CAUSE", "ALARM_DATE", "CONT_DATE", "DURATION_DAYS", "GIS_ACRES"],
        )
    if show_rx:
        rx.explore(
            m=m, name="Prescribed burns", color="blue",
            tooltip=["TREATMENT1", "START_DATE", "END_DATE", "GIS_ACRES"],
        )

    folium.LayerControl().add_to(m)
    return m

In [5]:
make_fire_map(fire_gdf, rxburn_gdf, "2025-01-01", "2025-12-31")

In [6]:
make_fire_map(fire_gdf, rxburn_gdf, "2020-01-01", "2021-12-31", causes=["Lightning"], show_rx=False)